# ViT / Swin matched-image geometry experiment

Executed locally on 2026-09-29 after the Colab sign-in service returned HTTP 502. Actual pretrained model outputs were used. The supplied notebook reproduces this run; its code cells are not marked as having executed in Colab.

## Data and observation definitions
- Identical 200 images to the previous CNN videos, 100 cats and 100 dogs. Archive paths, selected indices and label order match the ResNet-18 export exactly.
- Fixed ResNet-18 V1 preprocessing: resize 256, center crop 224, bilinear interpolation, ImageNet normalization. This controls preprocessing but is not each model's individually preferred evaluation recipe.
- Torchvision ImageNet-1K V1 weights, frozen evaluation mode, batch size 8.
- ViT-B/16: 12 blocks and explicit final layer normalization. CLS readout starts after block 1 because the input CLS token is identical for every image. Patch-mean readout also includes the positional input, and excludes CLS when pooling.
- Swin-T: stem, every attention block, each patch merging module, and final normalization. Pool H and W from NHWC block outputs.

## Measurements
S = mean between-class cosine distance / balanced mean within-class cosine distance, with each image vector L2-normalized. Within-class pairs exclude self-pairs.
Adjacent linear CKA compares centered sample Gram matrices of RAW pooled vectors.
PR = (sum eigenvalues)^2 / sum squared eigenvalues of the centered unit-vector sample Gram matrix.
Fisher = squared raw class-centroid distance / sum of the two within-class mean squared spreads.
No linear probe is fitted.

## Animation
Each image's cosine similarities to the same ordered 200-image anchors form a 200-coordinate fingerprint. Center each layer's fingerprints, fit one PCA across all observations for each readout, and use fixed 2D axes within that video. PCA bases are fitted separately between videos; cross-video positions/absolute projected distances are not comparable. All circles at measurement endpoints are real projected vectors; interlayer motion is interpolated and is not a measured continuous neural process.
The two lowest adjacent CKA values in each readout receive more animation time. These are descriptive minima, not statistical detection or evidence of a thermodynamic phase transition. Explicit final normalization is a readout change, not an extra attention block.

## Observed results
| Readout | Observations | First S | Last S | Lowest CKA | At |
|---|---:|---:|---:|---:|---|
| vit_b16_cls | 13 | 1.015 | 1.221 | 0.789 | b2 |
| vit_b16_patchmean | 14 | 1.025 | 1.286 | 0.342 | b1 |
| swin_t | 17 | 1.029 | 1.324 | 0.376 | s4.b1 |

S is higher at the final observation in every readout, but the trajectories are not monotonic. PR rises and then falls in the middle/late observations. ViT patch-mean and CLS expose different geometry, even within the same frozen network. Do not treat the cross-architecture comparison as a controlled test of depth alone: checkpoint training, architecture, and pooling differ.

## Reproduce
In Colab, open Attention_Model_Geometry_Matched_Images.ipynb and run all cells. On a machine with dependencies installed, run:

```bash
python run_attention_geometry.py
python analyze_attention_geometry.py
python make_attention_video.py vit_b16_cls
python make_attention_video.py vit_b16_patchmean
python make_attention_video.py swin_t
```

Implementation references: https://docs.pytorch.org/vision/0.22/_modules/torchvision/models/vision_transformer.html and https://docs.pytorch.org/vision/0.19/_modules/torchvision/models/swin_transformer.html
Dataset: https://zenodo.org/records/5226945 (archive MD5 is checked).


In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable,"-m","pip","install","numpy","pandas","matplotlib","scikit-learn","torch","torchvision"])

In [ ]:
from pathlib import Path
Path('run_attention_geometry.py').write_text('"""Matched-image ViT-B/16 and Swin-T geometry. Run in Python or Colab.\n\nDependencies: torch, torchvision, numpy. Uses the same archive, seed, and\nResNet-18 V1 preprocessing as the preceding CNN experiment.\n"""\nimport os\nimport sys\nfrom pathlib import Path\nROOT = Path(__file__).resolve().parent\nsys.path.append(str(ROOT / \'vision_deps\'))\nos.environ.setdefault(\'TORCH_HOME\', str(ROOT / \'model_cache\'))\nimport gc\nimport hashlib\nimport json\nimport time\nimport urllib.request\nimport zipfile\nfrom io import BytesIO\nimport numpy as np\nfrom PIL import Image\nimport torch\nimport torchvision\nfrom torchvision import models\nfrom torch.utils.data import Dataset, DataLoader\n\ndef dataset():\n    archive = ROOT / \'cats_dogs_light.zip\'\n    if not archive.exists():\n        print(\'Downloading matched dataset\', flush=True)\n        urllib.request.urlretrieve(\'https://zenodo.org/records/5226945/files/cats_dogs_light.zip?download=1\', archive)\n    assert hashlib.md5(archive.read_bytes()).hexdigest() == \'5e014163374c3bf7069c923de2d619c8\'\n    transform = models.ResNet18_Weights.IMAGENET1K_V1.transforms()\n    class Images(Dataset):\n        def __init__(self):\n            with zipfile.ZipFile(archive) as z:\n                self.names = [n for n in z.namelist() if \'/test/\' in n.lower()\n                              and n.lower().endswith((\'.jpg\', \'.jpeg\', \'.png\'))\n                              and Path(n).name.lower().startswith((\'cat.\', \'dog.\'))]\n            targets = np.array([3 if Path(n).name.lower().startswith(\'cat.\') else 5 for n in self.names])\n            rng = np.random.default_rng(42)\n            self.indices = np.concatenate([rng.choice(np.flatnonzero(targets == c), 100, replace=False) for c in (3, 5)])\n            self.indices = rng.permutation(self.indices)\n            self.labels = targets[self.indices]\n            self.paths = np.array([self.names[i] for i in self.indices])\n            reference = ROOT / \'resnet18_per_image_vectors.npz\'\n            if reference.exists():\n                with np.load(reference) as a:\n                    assert np.array_equal(a[\'archive_paths\'], self.paths)\n                    assert np.array_equal(a[\'labels\'], self.labels)\n        def __len__(self): return 200\n        def __getitem__(self, j):\n            with zipfile.ZipFile(archive) as z:\n                image = Image.open(BytesIO(z.read(self.paths[j]))).convert(\'RGB\')\n            return transform(image)\n    return Images(), str(transform)\n\ndef extract(model, observations, loader, device, readers):\n    record = {r: {name: [] for name, _ in observations} for r in readers}\n    hooks = []\n    for name, module in observations:\n        def capture(m, inp, out, name=name):\n            for r, pool in readers.items():\n                record[r][name].append(pool(out.detach()).cpu().numpy())\n        hooks.append(module.register_forward_hook(capture))\n    start = time.time()\n    try:\n        model.eval().to(device)\n        with torch.inference_mode():\n            for j, batch in enumerate(loader):\n                model(batch.to(device))\n                print(f\'  batch {j+1}/{len(loader)}, {time.time()-start:.1f}s\', flush=True)\n    finally:\n        for hook in hooks: hook.remove()\n        model.cpu()\n        if device.type == \'cuda\': torch.cuda.empty_cache()\n    return {r: {k: np.concatenate(v) for k, v in feats.items()} for r, feats in record.items()}\n\ndef save(slug, model_name, readout, feats, data, transform, weight):\n    # Before attention the CLS token is identical across images, so cosine\n    # geometry/CKA is degenerate. Exclude it from this readout, explicitly.\n    if readout == \'CLS\': feats.pop(\'tokens+pos\', None)\n    payload = dict(stages=np.array(list(feats)), labels=data.labels,\n                   archive_paths=data.paths, selected_indices=data.indices,\n                   model=np.array(model_name), readout=np.array(readout),\n                   weights=np.array(weight), preprocessing=np.array(transform))\n    for j, (name, x) in enumerate(feats.items()):\n        assert x.ndim == 2 and x.shape[0] == 200 and np.isfinite(x).all(), (name, x.shape)\n        payload[f\'features_{j}\'] = x.astype(np.float32)\n    np.savez_compressed(ROOT / f\'{slug}_per_image_vectors.npz\', **payload)\n    print(\'Saved\', slug, len(feats), \'observations\', flush=True)\n\ndef main():\n    torch.manual_seed(42)\n    torch.set_num_threads(min(4, os.cpu_count() or 1))\n    device = torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n    data, transform = dataset()\n    loader = DataLoader(data, batch_size=8, shuffle=False, num_workers=0)\n    print(\'Matched images:\', len(data), \'device:\', device, flush=True)\n    print(\'Starting ViT-B/16\', flush=True)\n    model = models.vit_b_16(weights=models.ViT_B_16_Weights.IMAGENET1K_V1)\n    obs = [(\'tokens+pos\', model.encoder.dropout)]\n    obs += [(f\'b{i+1}\', b) for i, b in enumerate(model.encoder.layers)]\n    obs += [(\'final_norm\', model.encoder.ln)]\n    feats = extract(model, obs, loader, device,\n                    {\'CLS\': lambda h: h[:, 0, :], \'patch_mean\': lambda h: h[:, 1:, :].mean(1)})\n    save(\'vit_b16_cls\', \'ViT-B/16\', \'CLS\', feats[\'CLS\'], data, transform, \'IMAGENET1K_V1\')\n    save(\'vit_b16_patchmean\', \'ViT-B/16\', \'patch_mean\', feats[\'patch_mean\'], data, transform, \'IMAGENET1K_V1\')\n    del model, feats, obs\n    gc.collect()\n    print(\'Starting Swin-T\', flush=True)\n    model = models.swin_t(weights=models.Swin_T_Weights.IMAGENET1K_V1)\n    obs = []\n    for i, m in enumerate(model.features):\n        if i == 0: obs.append((\'stem\', m))\n        elif i % 2 == 0: obs.append((f\'merge{i//2}\', m))\n        else: obs.extend((f\'s{(i+1)//2}.b{j+1}\', b) for j, b in enumerate(m))\n    obs.append((\'final_norm\', model.norm))\n    feats = extract(model, obs, loader, device, {\'spatial_mean\': lambda h: h.mean((1, 2))})\n    save(\'swin_t\', \'Swin-T\', \'spatial_mean\', feats[\'spatial_mean\'], data, transform, \'IMAGENET1K_V1\')\n    manifest = dict(torch=torch.__version__, torchvision=torchvision.__version__, device=str(device),\n                    seed=42, cohort_size=200, class_counts={\'cat\':100,\'dog\':100},\n                    transform=transform, output_basis=\'raw pooled block outputs; final_norm is explicit\',\n                    cls_start=\'b1; constant input CLS excluded\',\n                    caveat=\'Fixed ResNet-18 transform across architectures; checkpoint recipes differ.\')\n    (ROOT / \'attention_geometry_provenance.json\').write_text(json.dumps(manifest, indent=2))\n\nif __name__ == \'__main__\': main()\n')


In [ ]:
from pathlib import Path
Path('analyze_attention_geometry.py').write_text('"""Geometry metrics and descriptive plots for matched-image attention models."""\nfrom pathlib import Path\nimport json\nimport numpy as np\nimport pandas as pd\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\n\nROOT=Path(__file__).resolve().parent\nSLUGS=(\'vit_b16_cls\',\'vit_b16_patchmean\',\'swin_t\')\n\ndef stats(x,y):\n    x=np.asarray(x,dtype=np.float64)\n    z=x/np.maximum(np.linalg.norm(x,axis=1,keepdims=True),1e-12)\n    a,b=z[y==3],z[y==5]\n    def wc(q):\n        n=len(q)\n        return 1-(np.square(q.sum(0)).sum()-n)/(n*(n-1))\n    within=(wc(a)+wc(b))/2\n    between=1-a.mean(0)@b.mean(0)\n    ac,bc=x[y==3],x[y==5]\n    ma,mb=ac.mean(0),bc.mean(0)\n    spread=np.square(ac-ma).sum(1).mean()+np.square(bc-mb).sum(1).mean()\n    centered=z-z.mean(0)\n    ev=np.maximum(np.linalg.eigvalsh(centered@centered.T),0)\n    return dict(d_within_cos=within,d_between_cos=between,S=between/within,\n                Fisher_raw=np.square(ma-mb).sum()/max(spread,1e-20),\n                PR=ev.sum()**2/max(np.square(ev).sum(),1e-20))\n\ndef gram(x):\n    x=x.astype(np.float64);x=x-x.mean(0)\n    k=x@x.T\n    return k/max(np.linalg.norm(k),1e-20)\n\ndef main():\n    rows=[]\n    ref=np.load(ROOT/\'resnet18_per_image_vectors.npz\')\n    for slug in SLUGS:\n        a=np.load(ROOT/f\'{slug}_per_image_vectors.npz\')\n        assert np.array_equal(a[\'archive_paths\'],ref[\'archive_paths\'])\n        assert np.array_equal(a[\'labels\'],ref[\'labels\'])\n        prev=None\n        for i,stage in enumerate(a[\'stages\']):\n            x=a[f\'features_{i}\'];g=gram(x)\n            rows.append(dict(slug=slug,model=str(a[\'model\']),readout=str(a[\'readout\']),\n                             depth=i,stage=str(stage),feature_width=x.shape[1],\n                             CKA_prev=np.nan if prev is None else np.sum(g*prev),\n                             **stats(x,a[\'labels\'])))\n            prev=g\n    df=pd.DataFrame(rows)\n    df.to_csv(ROOT/\'attention_geometry_metrics.csv\',index=False)\n    summary=[]\n    for slug,g in df.groupby(\'slug\',sort=False):\n        i=g.CKA_prev.idxmin()\n        summary.append(dict(slug=slug,observations=len(g),S_first=float(g.S.iloc[0]),\n                            S_last=float(g.S.iloc[-1]),min_CKA=float(g.loc[i,\'CKA_prev\']),\n                            min_CKA_at=g.loc[i,\'stage\'],PR_last=float(g.PR.iloc[-1]),\n                            Fisher_last=float(g.Fisher_raw.iloc[-1])))\n    (ROOT/\'attention_geometry_summary.json\').write_text(json.dumps(summary,indent=2))\n    print(json.dumps(summary,indent=2),flush=True)\n    fig,axs=plt.subplots(3,2,figsize=(13,10),constrained_layout=True)\n    fields=((\'CKA_prev\',\'Adjacent linear CKA (raw vectors)\'),(\'S\',\'Relative separation S\'),\n            (\'d_within_cos\',\'Within-class cosine distance\'),(\'d_between_cos\',\'Between-class cosine distance\'),\n            (\'PR\',\'Participation ratio (unit vectors)\'),(\'Fisher_raw\',\'Fisher ratio (raw vectors)\'))\n    for ax,(f,title) in zip(axs.flat,fields):\n        for slug,g in df.groupby(\'slug\',sort=False):\n            ax.plot(g.depth,g[f],\'-o\',ms=3,label=f\'{g.model.iloc[0]} / {g.readout.iloc[0]}\')\n        ax.set(title=title,xlabel=\'Observed output index (not equal architectural depth)\')\n        ax.grid(alpha=.22)\n        ax.legend(fontsize=8)\n    fig.suptitle(\'Same 200 cat/dog images; fixed preprocessing; descriptive cross-architecture comparison\',fontsize=13)\n    fig.savefig(ROOT/\'attention_geometry_metrics.png\',dpi=140)\n\nif __name__==\'__main__\':main()\n')


In [ ]:
from pathlib import Path
Path('make_attention_video.py').write_text('"""Animate actual per-image ConvNeXt-Tiny geometry with time dilation at CKA dips.\n\nThe shared 2D PCA is fitted to every image\'s similarity fingerprint against\nthe same 200 images, pooled over all observed blocks. Motion between measured\nblocks is an illustration; only the endpoints are actual model outputs.\n"""\nfrom pathlib import Path\n\nimport matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nfrom matplotlib.animation import FFMpegWriter, FuncAnimation\nimport numpy as np\nimport sys\nimport pandas as pd\nfrom sklearn.decomposition import PCA\n\nROOT = Path(__file__).resolve().parent\nslug = sys.argv[1]\ndf = pd.read_csv(ROOT / "attention_geometry_metrics.csv")\ndf = df.loc[df.slug.eq(slug)].reset_index(drop=True)\nstages = df.stage.to_list()\nsource = np.load(ROOT / f"{slug}_per_image_vectors.npz")\nlabels = source["labels"]\nassert np.array_equal(source["stages"], stages)\nL = len(stages)\nmodel_title = str(source["model"]) + " / " + str(source["readout"])\nassert np.array_equal(np.unique(labels, return_counts=True)[1], [100, 100])\n\n# Channel counts vary between stages. A same-image 200-coordinate relational\n# fingerprint gives a comparable feature space for one shared PCA fit.\nprofiles = []\nfor k in range(len(stages)):\n    raw = source[f"features_{k}"].astype(np.float64)\n    unit = raw / np.maximum(np.linalg.norm(raw, axis=1, keepdims=True), 1e-12)\n    sim = unit @ unit.T\n    profiles.append(sim - sim.mean(axis=0, keepdims=True))\nprofiles = np.asarray(profiles)\npca = PCA(n_components=2, svd_solver="full")\nclouds = pca.fit_transform(profiles.reshape(-1, 200)).reshape(L, 200, 2)\nprint("Shared PCA explained variance:", pca.explained_variance_ratio_, flush=True)\n\nx = np.arange(L)\nwithin = df.d_within_cos.to_numpy()\nbetween = df.d_between_cos.to_numpy()\nseparation = df.S.to_numpy()\ncka = df.CKA_prev.to_numpy()\n\nFPS, SECONDS = 20, 12\nFRAMES = FPS * SECONDS\n# Segment j is the passage from observation j-1 to j. Slower segments have\n# more screen time, but the measurement positions on the plots remain discrete.\nfocus = np.argsort(cka[1:])[:2] + 1\nweights = np.full(L-1, .45)\nfor j in focus: weights[j-1] = 1.8 if j != focus[0] else 2.7\nstage_time = np.r_[0, np.cumsum(weights)]\nstage_time *= SECONDS / stage_time[-1]\nprint("Slow intervals (s):", {stages[i]: round(stage_time[i] - stage_time[i-1], 2)\n                              for i in focus}, flush=True)\n\nplt.rcParams.update({\n    "font.family": "DejaVu Sans", "font.size": 11,\n    "text.color": "#eef3fc", "axes.labelcolor": "#aab8ce",\n    "xtick.color": "#9aabc0", "ytick.color": "#9aabc0",\n})\nfig = plt.figure(figsize=(14.4, 8.1), dpi=100, facecolor="#0b1424")\nax = fig.add_axes([.057, .20, .49, .60], facecolor="#111f33")\nax_s = fig.add_axes([.63, .62, .32, .18], facecolor="#111f33")\nax_c = fig.add_axes([.63, .39, .32, .17], facecolor="#111f33")\nax_d = fig.add_axes([.63, .20, .32, .14], facecolor="#111f33")\nfor a in (ax, ax_s, ax_c, ax_d):\n    for spine in a.spines.values():\n        spine.set_color("#34475e")\n    a.grid(color="#52637c", alpha=.22, lw=.7)\n\nlimits = np.array([[clouds[:, :, j].min(), clouds[:, :, j].max()]\n                   for j in range(2)])\npadding = .08 * (limits[:, 1] - limits[:, 0])\nax.set(xlim=(limits[0, 0]-padding[0], limits[0, 1]+padding[0]),\n       ylim=(limits[1, 0]-padding[1], limits[1, 1]+padding[1]),\n       xlabel="Shared PCA 1  ·  similarity fingerprint", ylabel="Shared PCA 2")\nax.set_aspect("equal", adjustable="box")\nax.set_title("200 real image vectors  ·  shared projection", loc="left",\n             pad=12, weight="bold", fontsize=16)\ncolors = ["#65d7da", "#ff9f79"]\nscats = [ax.scatter([], [], s=25, c=color, alpha=.76, lw=0, label=name)\n         for color, name in zip(colors, ["cat (100)", "dog (100)"])]\ncenters = [ax.scatter([], [], s=180, c=color, marker="x", lw=2.6)\n           for color in colors]\nax.legend(loc="upper right", frameon=False, fontsize=10, ncol=2)\n\nfor a in (ax_s, ax_c, ax_d):\n    a.set_xlim(-.5, L-.5)\n    for boundary in focus:\n        a.axvline(boundary, color="#f2c56b", alpha=.30, lw=1.2)\n    a.set_xticks(sorted(set([0, L-1, *focus, *range(0,L,4)])))\nax_s.plot(x, separation, color="#ccb1ff", lw=2.5, marker="o", ms=3)\ndot_s, = ax_s.plot([], [], "o", ms=10, color="#ccb1ff", mec="white", mew=1.2)\nax_s.set(ylim=(min(.98, separation.min()-.03), max(1.39, separation.max()+.03)), ylabel="S", title="Relative separation  ·  between / within")\nax_s.tick_params(axis="x", labelbottom=False)\nax_c.plot(x[1:], cka[1:], color="#f4c778", lw=2.2, marker="o", ms=3)\nfor j in focus:\n    ax_c.scatter([j], [cka[j]], s=90, facecolor="none", edgecolor="#ff7c8a", lw=1.6, zorder=5)\ndot_c, = ax_c.plot([], [], "o", ms=10, color="#f4c778", mec="white", mew=1.2)\nax_c.set(ylim=(0, 1.05), ylabel="CKA", title="Adjacent CKA  ·  measured at each endpoint")\nax_c.tick_params(axis="x", labelbottom=False)\nax_d.plot(x, within, color=colors[0], lw=2.1, label="within")\nax_d.plot(x, between, color=colors[1], lw=2.1, label="between")\ndot_w, = ax_d.plot([], [], "o", ms=8, color=colors[0])\ndot_b, = ax_d.plot([], [], "o", ms=8, color=colors[1])\nax_d.set(ylim=(0, max(between.max(), within.max())*1.1), ylabel="Cosine distance", xlabel="Observed block / transition")\nax_d.legend(frameon=False, loc="upper left", fontsize=9, ncol=2)\n\nfig.text(.057, .94, model_title + "  /  cat versus dog", fontsize=23, weight="bold")\nfig.text(.057, .89, f"ImageNet pretrained  ·  {L} observed outputs  ·  same 200 images and preprocessing",\n         fontsize=12, color="#b6c5d9")\nlayer_text = fig.text(.057, .135, "", fontsize=19, weight="bold")\nevent_text = fig.text(.057, .091, "", fontsize=13, color="#f4c778")\nmetric_text = fig.text(.63, .865, "", fontsize=13, color="#e0e9f6")\nfig.text(.057, .034,\n         "Measured dots use one shared PCA of same-image cosine profiles; motion is interpolated. A CKA dip signals reorganization, not proof of a phase transition.",\n         fontsize=9.6, color="#aabbd0")\nprogress = fig.add_axes([.057, .066, .893, .006], facecolor="#24364b")\nprogress.set(xlim=(0, 1), ylim=(0, 1))\nprogress.set_axis_off()\nbar = progress.barh(.5, 0, height=1, color="#f4c778", left=0)[0]\n\ndef update(frame):\n    t = frame / (FRAMES-1) * SECONDS\n    q = float(np.interp(t, stage_time, x))\n    i = min(int(q), len(stages)-2)\n    blend = q - i\n    eased = 3*blend**2 - 2*blend**3\n    pts = (1-eased)*clouds[i] + eased*clouds[i+1]\n    for j, label in enumerate((3, 5)):\n        selected = pts[labels == label]\n        scats[j].set_offsets(selected)\n        centers[j].set_offsets(selected.mean(axis=0))\n    w = np.interp(q, x, within)\n    b = np.interp(q, x, between)\n    s = b/w\n    dot_s.set_data([q], [s])\n    dot_w.set_data([q], [w])\n    dot_b.set_data([q], [b])\n    current = min(round(q), L-1)\n    target = min(int(np.ceil(q)), L-1)\n    if q == 0:\n        dot_c.set_data([], [])\n    else:\n        dot_c.set_data([target], [cka[target]])\n    layer_text.set_text(f"{current+1:02d} / {L:02d}    {stages[current]}")\n    metric_text.set_text(f"S  {s:.3f}   ·   within  {w:.3f}   ·   between  {b:.3f}")\n    if target in focus and q < target:\n        event_text.set_text(f"SLOW MOTION   {stages[target-1]} → {stages[target]}   ·   adjacent CKA {cka[target]:.3f}")\n        event_text.set_color("#ffb879")\n    else:\n        event_text.set_text("Lower adjacent CKA = stronger change in the 200-image relational geometry")\n        event_text.set_color("#aabbd0")\n    bar.set_width((frame+1)/FRAMES)\n    return (*scats, *centers, dot_s, dot_c, dot_w, dot_b,\n            layer_text, event_text, metric_text, bar)\n\nanimation = FuncAnimation(fig, update, frames=FRAMES, interval=1000/FPS, blit=False)\nout = ROOT / f"{slug}_geometry_slow_12s.mp4"\nanimation.save(out, writer=FFMpegWriter(fps=FPS, codec="libx264", bitrate=3000,\n                                      extra_args=["-pix_fmt", "yuv420p", "-movflags", "+faststart"]))\nupdate(int(stage_time[focus[0]]/SECONDS*(FRAMES-1))-12)\nfig.savefig(ROOT / f"{slug}_geometry_preview.png", facecolor=fig.get_facecolor(), dpi=100)\nprint("Video:", out)\n')


In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable,"run_attention_geometry.py"])
subprocess.check_call([sys.executable,"analyze_attention_geometry.py"])

In [ ]:
from IPython.display import display, Image
import pandas as pd
display(pd.read_csv("attention_geometry_metrics.csv"))
display(Image(filename="attention_geometry_metrics.png"))

In [ ]:
import sys, subprocess
from IPython.display import Video, display
for slug in ("vit_b16_cls","vit_b16_patchmean","swin_t"):
    subprocess.check_call([sys.executable,"make_attention_video.py",slug])
    display(Video(f"{slug}_geometry_slow_12s.mp4",embed=True))